# System Architecture and Communication Topology
---

A training step is a tour through your machine's bandwidth hierarchy.  Tensors live in GPU memory (HBM), travel between GPUs over NVLink or PCIe, get pulled from host memory across PCIe, and — in multi-node training — go between nodes over the network.  Each layer has a different speed, and any one of them can be the bottleneck.

By the end of this notebook you will have:

- A concrete sense of the bandwidth differences between HBM, NVLink, PCIe, host memory, and the NIC.
- A working mental model of the physical topology of a multi-GPU node and how that topology maps onto NCCL collective operations.
- The vocabulary to recognize, in an nsys timeline, which level of the hierarchy is currently bottlenecking your training step.

## Important Terminology

- **Host**: The CPU and its memory (host memory).
- **Device**: The GPU and its memory (device memory; "HBM" for the high-bandwidth memory on data-center GPUs).
- **Kernel**: A GPU function executed on the device.
- **Latency**: The time it takes for a unit of data to travel from point A to point B.
- **Bandwidth**: The amount of data that can flow between two points per unit time.  This notebook is almost entirely about bandwidth.

## The Bandwidth Hierarchy

Here is the rough bandwidth picture you should carry in your head when reading a profile.  All numbers are per-GPU and bidirectional (this is how NVIDIA quotes them).

| Layer | What it carries | H100 SXM (8× node) | 4× L4 (cloud node) |
|---|---|---|---|
| **HBM** | inside-GPU memory ↔ SM | ~3 TB/s | ~300 GB/s |
| **NVLink** | GPU ↔ GPU within node | 900 GB/s | none — PCIe only |
| **PCIe Gen5 ×16** | CPU ↔ GPU | ~128 GB/s | typically Gen4 ×16 (~64 GB/s) |
| **Host DRAM** | CPU ↔ RAM | ~300–500 GB/s | similar |
| **NIC** | node ↔ node | 400 Gb/s (~50 GB/s) | typically 100 Gb/s (~12 GB/s) |

Two things worth absorbing from this table:

1. **There are three orders of magnitude between the top of this table and the bottom.**  Reading a 1 GB tensor from HBM on the same GPU takes about a third of a millisecond.  Moving that same tensor between nodes over a 400 Gb/s NIC takes about 20 ms — sixty times longer.  Every bandwidth layer you cross adds time, and the layers are not close to each other in speed.
2. **The hierarchy is different on different hardware.**  The "no NVLink" row for L4 isn't a footnote — it changes the physical paths every NCCL collective takes.  An AllReduce on H100 SXM runs at NVLink speed; on a 4× L4 node it runs at PCIe speed, which is roughly an order of magnitude slower.  *Same code, same model, different bottleneck.*

The profiler's job in this lab is to teach you to spot which layer is currently slowing you down — and the rest of the workshop's labs are essentially detailed worked examples of each.

## Communication Approaches

Before we look at real hardware, two basic patterns for moving data between GPUs:

### I. Host Staging of Copy Operations

The diagram below illustrates a GPU-to-GPU memory copy that takes the data through host memory.  Data leaves `GPU 0` across the PCIe bus, gets staged in a CPU buffer, then gets copied to `GPU 1`.  Two PCIe traversals and a CPU round-trip — slower than necessary, and the wrong choice whenever a direct GPU-to-GPU path is available.

<img src="images/DL_host_staging.png" width="380px" height="380px" alt-text="Host staging"/>

### II. Peer-to-Peer Memory Access

Peer-to-peer (P2P) lets GPUs read and write each other's memory directly, either over NVLink (fast path) or over PCIe (slower fallback path).  No CPU round-trip.  This is what NCCL uses on a properly configured node.

<img src="images/DL_p2p.png" width="380px" height="380px" alt-text="P2P"/>

P2P requires a Unified Virtual Address Space (UVA) — a single address space shared by the host and all GPUs.  That's been standard since Compute Capability 2.0, so any data-center GPU you'll touch supports it.

Let's see what P2P paths are actually available on your node.

## Discovering the Topology

Run the next cell to see which GPU pairs can do P2P reads:

In [ ]:
!nvidia-smi topo -p2p r

**Expected output on H100 SXM** (the hardware you're profiling on):

<img src="images/H100-p2p-r.png" width="380px" height="380px" alt-text="p2p reads"/>

All GPU pairs can do P2P reads.  Now check which pairs can use NVLink specifically:

In [ ]:
!nvidia-smi topo -p2p n

**Expected output on H100 SXM**:

<img src="images/H100-p2p-n.png" width="380px" height="380px" alt-text="p2p nvlink"/>

All pairs can use NVLink for P2P, which means host staging is unnecessary and every GPU-to-GPU collective will run at NVLink speed.

**For comparison — what the same commands return on a 4× L4 cloud node**: the `-p2p n` matrix would show `NS` (Not Supported) for every GPU pair, because L4 has no NVLink hardware at all.  P2P over PCIe is still possible between GPUs sharing a PCIe switch (the `-p2p r` matrix would show some `OK` entries), but cross-switch traffic falls back to host staging.  Same `nvidia-smi` command, very different story about what your collectives are going to cost.

The `-p2p` flag takes one of these capability codes:

- **r**: P2P read capability
- **w**: P2P write capability
- **n**: P2P NVLink capability
- **a**: P2P atomic capability
- **p**: P2P prop capability

## Node Topology in Detail

For the full GPU-and-NIC topology of the node, use `topo -m`:

In [ ]:
!nvidia-smi topo -m

**Expected output on H100 SXM (8-GPU node)**:

<center><img src="images/dgx-h100-terminal-topo.png" width="850px" alt-text="h100 topo"/></center>

Every GPU-pair cell reads `NV18`: each GPU is connected to every other GPU over 18 NVLink lanes, giving the full 900 GB/s of GPU-to-GPU bandwidth.  At the system level this is the topology:

<center><img src="images/dgxh100-topo.png" width="700px" alt-text="dgx topology"/></center>

All 8 GPUs hang off a common NVSwitch fabric.  Any GPU can talk to any other at full NVLink speed in one hop.

**For comparison — a 4× L4 cloud node** typically returns `PHB` or `SYS` for most GPU pairs in the same matrix.  `PHB` means traffic crosses the PCIe Host Bridge (PCIe switch + CPU root complex); `SYS` means it crosses the inter-CPU interconnect (UPI/Infinity Fabric) as well.  Both are PCIe-bandwidth at best, and `SYS` adds an extra hop through the CPU socket interconnect — measurably slower again.  This single matrix gives you an immediate read on what your collectives are going to cost on whatever hardware you happen to be running on.

## NVLink and NVLink Switch

NVLink is the high-bandwidth GPU-to-GPU interconnect that makes server-class multi-GPU training fast.  An H100 SXM has 18 NVLink lanes for 900 GB/s of bidirectional bandwidth per GPU.  NVSwitch chips connect those lanes into a fully-connected fabric, so every GPU is one hop from every other.

<center><img src="images/nvlink.png" width="850px" alt-text="nvlink"/></center>
<center><img src="images/nvlink-nvswitch.png" width="850px" alt-text="nvlink-nvswitch"/></center>

Beyond a single node, NVLink Switch chips can extend the fabric to up to 256 GPUs (NVLink Network) — but that's hardware most workshops won't have access to.  In-node NVLink is the universal case for the labs that follow.

## NCCL: Where Your Collectives Actually Run

The [NVIDIA Collective Communications Library](https://developer.nvidia.com/nccl) (NCCL) provides GPU-to-GPU communication primitives that are topology-aware.  When you call `AllReduce` in your training script, NCCL is the thing deciding which physical paths each chunk of data takes.  Understanding which path each collective stresses is the bridge between this topology lab and what you'll see in an nsys timeline in the rest of the workshop.

NCCL's primitives:

- **AllReduce** — sum/min/max across all ranks; result on every rank.  In the ring algorithm: each chunk of data traverses every GPU twice (a reduce-scatter pass followed by an all-gather pass).  Bottleneck: **NVLink intra-node, NIC inter-node.**

<center><img src="images/allreduce.png" width="380px" alt-text="allreduce"/></center>

- **Broadcast** — one rank's buffer copied to all others.  Naive cost = sequential copies; tree algorithm gets it to `log(k)` hops.  Bottleneck: same path as AllReduce.

<center><img src="images/broadcast.png" width="380px" alt-text="broadcast"/></center>

- **Reduce** — sum/min/max across all ranks; result only on one root rank.  Inverse of Broadcast in cost terms.

<center><img src="images/reduce.png" width="380px" alt-text="reduce"/></center>

- **AllGather** — gather `N` values from each of `k` ranks into a `k·N`-element buffer on every rank.  Bottleneck: same as AllReduce (it's literally the second half of AllReduce).

<center><img src="images/allgather.png" width="380px" alt-text="allgather"/></center>

- **ReduceScatter** — sum/min/max across all ranks; each rank gets one chunk of the result.  Bottleneck: same.  Together with AllGather it makes up AllReduce.

<center><img src="images/reducescatter.png" width="380px" alt-text="reducescatter"/></center>

NCCL also supports point-to-point send/receive for scatter / gather / all-to-all patterns.

**The key insight for profiling**: every NCCL operation has a *fastest possible* time, set by the bandwidth of its bottleneck path.  AllReduce of an `N`-byte tensor on a `k`-GPU ring is bounded below by `2 · N · (k-1)/k / bandwidth`.  If nsys shows your AllReduce taking dramatically longer than this floor, something is wrong — usually CPU-side launch overhead, sync points inserted by debug flags, or a fallback to a slower physical path (host staging when P2P fails).

### NCCL Environment Variables

NCCL has an extensive set of [environment variables](https://docs.nvidia.com/deeplearning/nccl/user-guide/docs/env.html) for tuning and debugging.  The handful worth knowing right now:

- **`NCCL_DEBUG`** — controls log verbosity.  `NCCL_DEBUG=INFO` prints algorithm selection at startup and is the single most useful debugging knob: it tells you which physical paths NCCL chose for each collective.
- **`NCCL_SOCKET_IFNAME`** — restrict NCCL to specific network interfaces (e.g. `NCCL_SOCKET_IFNAME==ens1f0`, or `NCCL_SOCKET_IFNAME=^docker` to exclude container bridges).
- **`NCCL_IB_DISABLE=1`** — turn off InfiniBand/RoCE and fall back to TCP sockets.  Useful for sanity-checking that IB is actually being used: compare timings with and without.
- **`NCCL_P2P_DISABLE=1`** — disable peer-to-peer entirely, forcing host staging.  Useful for measuring the speedup P2P is buying you on a given platform.
- **`NCCL_ALGO`** — pin a specific algorithm (`ring`, `tree`, `nvls`, …).  Mostly for performance debugging; NCCL's auto-selection is usually right.
- **`NCCL_P2P_LEVEL`** — fine-grained control over when NCCL uses P2P vs host staging.  Cutoff codes: `LOC`/0 (never), `NVL` (only over NVLink), `PIX`/1 (same PCIe switch), `PXB`/2 (across PCIe switches), `PHB`/3 (same NUMA node, via CPU), `SYS`/4 (across NUMA).

## Identifying Bottlenecks with nsys

This is where the topology lab connects to the rest of the workshop.  In an nsys timeline, you'll see GPU streams along the top and CPU threads below.  The pattern of activity — and especially the *gaps* — tells you which layer of the bandwidth hierarchy is currently slowing you down.

| Symptom in the nsys timeline | Likely bottleneck |
|---|---|
| Big gaps in GPU activity between kernel launches; CPU thread looks busy | **CPU / kernel-launch overhead.**  You're not feeding the GPU fast enough.  Common when kernels are too small, or there are too many of them. |
| GPU mostly busy, but `Memcpy HtoD` / `Memcpy DtoH` bars between steps are wide | **PCIe (CPU↔GPU).**  You're moving data through PCIe each step.  Pin memory, preload to device, use `non_blocking=True`. |
| GPU mostly busy with kernels; NCCL collective bars are wide relative to compute | **NVLink intra-node, or NIC inter-node.**  Check `NCCL_DEBUG=INFO` to confirm which path NCCL chose.  Gradient accumulation or less-frequent communication can help. |
| Kernels themselves are slow relative to their FLOPs/byte; GPU is busy but not productive | **HBM.**  You're memory-bound on the GPU itself.  Larger batch, kernel fusion, or a different precision can help. |
| Backward-pass kernels separated by tiny GPU-idle gaps in a regular sawtooth pattern | **CPU-side sync** (e.g. `set_detect_anomaly(True)`, frequent `.item()` calls).  Not a bandwidth issue at all — just synchronizations forcing the GPU to wait on the CPU between every backward op. |
| GPU mostly idle, no kernels running, no memcpys; CPU thread spending time in `DataLoader` | **Host data pipeline** (covered in the intro lab — `num_workers`, `pin_memory`). |

Bookmark this table.  The labs that follow are essentially "here's what each of these looks like in detail, here's how to fix it."

## Next: Distributed Training Strategy

With the topology and bandwidth hierarchy in hand, the next lab applies these ideas to a real distributed training setup using data parallelism over NCCL.

## <center><div style="text-align:center; color:#FF0000; border:3px solid red;height:80px;"> <b><br/> [Next Notebook](data-parallelism.ipynb) </b> </div></center>

---
## References

- https://www.nvidia.com/en-us/data-center/nvlink/
- https://developer.nvidia.com/blog/nvidia-nvlink-and-nvidia-nvswitch-supercharge-large-language-model-inference
- https://docs.nvidia.com/deeplearning/nccl/user-guide/docs/env.html
- https://developer.nvidia.com/nsight-systems/get-started

## Licensing

Copyright © 2026 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). These materials include references to hardware and software developed by other entities; all applicable licensing and copyrights apply.